#### Imports

In [1]:
import os
from typing import Annotated, Sequence
from typing_extensions import TypedDict
 
from dotenv import load_dotenv
from langchain_core.messages import BaseMessage
from langchain_core.tools import tool
from langchain_anthropic import ChatAnthropic
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

#### env configuration

In [ ]:
# Load variables from your .env file
load_dotenv()
 
api_key = os.getenv("KEY")
base_url = os.getenv("BASE_URL")
model_name = os.getenv("MODEL")
 
# Initialize model through your custom lab gateway
llm = ChatAnthropic(
        model=model_name,
        api_key=api_key,
        anthropic_api_url=base_url,
        temperature=0
    )
print("Successfully initialized ChatAnthropic client.")


Successfully initialized ChatAnthropic client.


#### Defining the Mathematical Tools

In [3]:
@tool
def add_expenses(a: float, b: float) -> float:
    """Adds two expense amounts together and returns the sum."""
    return float(a + b)
 
@tool
def apply_tax(amount: float, tax_percent: float) -> float:
    """Applies a percentage tax (like GST) to a given amount and returns the total."""
    return float(amount * (1 + (tax_percent / 100)))
 
@tool
def split_bill(amount: float, people: int) -> float:
    """Splits a total bill amount equally among a specific number of people."""
    if people <= 0:
        raise ValueError("Number of people must be greater than zero.")
    return float(amount / people)
 
# Register tools and bind them to the initialized model wrapper
tools = [add_expenses, apply_tax, split_bill]
llm_with_tools = llm.bind_tools(tools)
print(f"Registered {len(tools)} operational agent tools.")

Registered 3 operational agent tools.


#### Building a cyclic graph flow (LangGraph)

In [5]:
# 1. Define the conversation state context
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
 
# 2. Define the LLM assessment behavior node
def call_model(state: AgentState):
    messages = state['messages']
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}
 
# 3. Assemble and build out the state graph workflow
workflow = StateGraph(AgentState)
 
# Append functional operational nodes
workflow.add_node("agent", call_model)
workflow.add_node("tools", ToolNode(tools))
 
# Establish tracking relationships and edge pathways
workflow.add_edge(START, "agent")
 
# Add conditional execution checks to route calls to tools or finish execution
workflow.add_conditional_edges(
    "agent",
    tools_condition,
)
 
# Pipe tool execution outcomes back into the main agent context
workflow.add_edge("tools", "agent")
 
# Compile runtime graph
app = workflow.compile()
print("Cyclic agent graph compiled successfully.")


Cyclic agent graph compiled successfully.


#### Running the sample tasks

In [7]:
def run_agent_task(prompt: str):
    print("\n" + "="*60)
    print(f"EXECUTING PROMPT: '{prompt}'")
    print("="*60)
    
    inputs = {"messages": [("user", prompt)]}
    
    # Stream the graph step execution to track calculation loops live
    for output in app.stream(inputs, stream_mode="values"):
        last_message = output["messages"][-1]
        last_message.pretty_print()
 
# Test Case 1: Single execution loop
run_agent_task("What is the total of 1200 and 800?")


EXECUTING PROMPT: 'What is the total of 1200 and 800?'
================================ Human Message =================================

What is the total of 1200 and 800?
================================== Ai Message ==================================

[{'id': 'toolu_bdrk_01FwNTZyVAZ4657yenN7pYHr', 'input': {'a': 1200, 'b': 800}, 'name': 'add_expenses', 'type': 'tool_use'}]
Tool Calls:
  add_expenses (toolu_bdrk_01FwNTZyVAZ4657yenN7pYHr)
 Call ID: toolu_bdrk_01FwNTZyVAZ4657yenN7pYHr
  Args:
    a: 1200
    b: 800
================================= Tool Message =================================
Name: add_expenses

2000.0
================================== Ai Message ==================================

The total of 1200 and 800 is **2000**.


In [8]:
# Test Case 2: Multi-step cyclic execution loop (Add -> Tax -> Split)
run_agent_task("Add 2000 and 1500, apply 18% tax, and split among 3 people.")


EXECUTING PROMPT: 'Add 2000 and 1500, apply 18% tax, and split among 3 people.'
================================ Human Message =================================

Add 2000 and 1500, apply 18% tax, and split among 3 people.
================================== Ai Message ==================================

[{'text': "I'll help you with that! Let me break this down into steps:\n\n1. First, add 2000 and 1500\n2. Then apply 18% tax to the sum\n3. Finally, split the total among 3 people", 'type': 'text'}, {'id': 'toolu_bdrk_01A4mAnTEbqwRx7KSiNJrgCj', 'input': {'a': 2000, 'b': 1500}, 'name': 'add_expenses', 'type': 'tool_use'}]
Tool Calls:
  add_expenses (toolu_bdrk_01A4mAnTEbqwRx7KSiNJrgCj)
 Call ID: toolu_bdrk_01A4mAnTEbqwRx7KSiNJrgCj
  Args:
    a: 2000
    b: 1500
================================= Tool Message =================================
Name: add_expenses

3500.0
================================== Ai Message ==================================

[{'text': 'Now let me apply the 18% tax